# Imports

In [ ]:
from cytools import Polytope

In [ ]:
import sys; sys.path.append('..')
from src import cydata, Zp, diagnostics

In [ ]:
import sys; sys.path.append('../../cornell-dev')
from projects.kklt.kklt_lib import kklt_conifolds

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time

Utility method

In [ ]:
def round_sig(x, sig=2):
    if x == 0:
        return 0
    return np.round(x, sig - np.floor(np.log10(np.abs(x))).astype(int) - 1)

# Load Example

In [ ]:
#verts   = [[1, 0, 0, 0], [0, 1, 0, 0], [-14, -9, -3, -1], [-3, -2, -1, 1], [0, 0, 0, 1], [0, 0, 1, 0], [-8, -5, -2, 0], [-4, -3, -1, 1], [-1, -1, 0, 1]]
#heights = [0.0, 0.0, 20.49999999999999, 0.0, -2.4999999999999964, 4.249999999999997, 6.749999999999995, 0.0, -7.999999999999995, -3.2499999999999982, -4.499999999999998, 7.249999999999999, -2.749999999999999, -5.249999999999997, 0.0]

#verts   = [[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0], [-32, -21, -9, -1], [-10, -7, -3, 1], [-3, -2, -1, 1], [-1, -1, 0, 1]]
#heights = [0.0, 0.0, 3.500000000000006, 0.0, 60.750000000000014, 51.750000000000014, -17.749999999999996, 0.0, -18.25, 16.500000000000014, -20.25, -20.750000000000004, 10.000000000000009, 4.5000000000000036, 24.250000000000007, 0.0, -0.24999999999999506, -3.5000000000000036, 36.0, -5.000000000000006, -3.500000000000005]

In [ ]:
# Manwe...
verts   = [[0, 0, 0, 0], [1, -1, -1, -1], [-1, 2, 1, 1], [-1, -1, 0, 0], [-1, -1, 2, 0], [-1, -1, 2, 1], [-1, 0, 0, 2], [-1, -1, 0, 2], [-1, 0, 0, 1], [-1, 0, 1, 0], [-1, -1, 0, 1], [-1, -1, 1, 0], [-1, -1, 1, 1], [-1, 0, 1, 1], [-1, 1, 1, 1], [0, -1, 0, 0]]
heights = [0, 35, 29, 35, 31, 35, 35, 35, 15, 17, 31, 9, 21]

Get the CY

In [ ]:
p  = Polytope(verts)
t  = p.triangulate(heights=heights)
cy = t.cy()

In [ ]:
(cy.h11(),cy.h21())

Get the conifold info

In [ ]:
conis = list(kklt_conifolds.kklt_conifolds(p.dual(), as_class=True))
assert len(conis) == 1
q = conis[0].conifold_charge()

Collect this into a "CYData" object

In [ ]:
data = cydata.CYData.from_cy(cy, coni_curve=q)

# Benchmark dilations

In [ ]:
dilations  = [1,2,3,4,5,7,10,13,16,20,25,30,35,40]
dilations += [50, 60, 70, 80, 90, 100]
dilations += [200, 300, 400, 500]
#dilations  = list(range(1,701+1,50))
#dilations  = [2000]
#dilations  = [20]
num_ps     = 1_000

Find the minimum degree p-vector (if the p-vectors are super far out, this speeds things up...)

In [ ]:
grading = np.sum(data.H_cob,axis=0)
mindegp = Zp.mindeg_pvec_gurobi(data)

mindeg  = np.dot(grading, mindegp)

In [ ]:
mindeg

Get some p-vectors

In [ ]:
t0 = time.time()
ps = Zp.pvecs(data, min_N_pts=num_ps, min_deg=mindeg, deg_window=mindeg//20, verbosity=1)
t1 = time.time()

print(f"Found {len(ps)} p-vectors in {t1-t0}s...")

if False:
    grading = data._H.sum(axis=0)
    grading = grading//np.gcd.reduce(grading)
    
    plt.hist(ps@grading[1:])
    plt.xlabel('degs')
    plt.ylabel('#')

Collect the dilation data

In [ ]:
dilation = []
num_pfvs = []
times    = []

for ellipsoid_dilation in dilations:
    print(f"Studying dilation={ellipsoid_dilation}...")
    t0 = time.time()
    pfvs = Zp.coniZpM(
        data=data,
        ps=ps,
        Qmax=data.h11+data.h21+4,
        M0min=13,
        #M0max=40,
        ellipsoid_dilation=ellipsoid_dilation,
        max_N_pfvs=1_000_000_000,
        verbosity=1,
        low_level_parallelism=1,
        return_formal_pfvs=True)
    t1 = time.time()

    dilation.append(ellipsoid_dilation)
    num_pfvs.append(len(pfvs))
    print(f"found {num_pfvs[-1]} PFVs...")
    times.append(t1-t0)

df = pd.DataFrame({'dilation': dilation, 'num_pfvs':num_pfvs, 'time':times})

print('DONE!', end=30*' ')

Plot the data

In [ ]:
num_fit_pts = 5

# make the figure
fig,axs = plt.subplots(3,1, sharex=True)

# raw data plots
# --------------
axs[0].scatter(df['dilation'], df['num_pfvs'])
axs[1].scatter(df['dilation'], df['time'])
axs[2].scatter(df['dilation'], df['num_pfvs']/df['time'])

# fits
# ----
if len(df) >= num_fit_pts >= 0:
    # #pfvs fit
    m,b = np.polyfit(
        x=df['dilation'][-num_fit_pts:],
        y=df['num_pfvs'][-num_fit_pts:],
        deg=1,
    )
    fit = m*df['dilation'] + b
    axs[0].plot(df['dilation'], fit,
                label=f'{round_sig(m)} * dilation + {round_sig(b)}')
    axs[0].legend()
    
    # time fit
    m,b = np.polyfit(
        x=np.log(df['dilation'][-num_fit_pts:]),
        y=np.log(df['time'][-num_fit_pts:]),
        deg=1,
    )
    fit = np.exp(b)*np.pow(df['dilation'],m)
    axs[1].plot(df['dilation'], fit,
                label=f'{round_sig(np.exp(b))} * dilation^{round_sig(m)}')
    axs[1].legend()

# yscales
# -------
#axs[0].set_yscale('log')
#axs[1].set_yscale('log')
#axs[2].set_yscale('log')

# various labels
# --------------
axs[0].set_ylabel('# PFVs')
axs[1].set_ylabel('time [s]')
axs[2].set_ylabel('PFV rate [$s^{-1}$]')
axs[2].set_xlabel('dilation')

title = ""
title += f"(h11,h21) = {data.h11,data.h21}; "
title += f"{len(ps)} p-vectors, dilation study"
axs[0].set_title(title)

In [ ]:
for pfv in pfvs:
    print(pfv.check_all(),end=',')
    if not pfv.check_all():
        for _ in range(10):
            print("AAAA BAD")

# Diagnose dilations

In [ ]:
step = 10

dilations_plt = [pfv.ellipsoid_required_dilation for pfv in pfvs]
plt.hist(dilations_plt, bins=np.arange(
    start=min(dilations_plt)-0.5,
    stop =max(dilations_plt)+0.5+step,
    step = step,
))
plt.yscale('log')
plt.xlabel('required dilation')
plt.ylabel('# PFV')
plt.title(f'Manwe studying {len(ps)} p-vectors up to dilation {max(dilations)}')

In [ ]:
ps_safe  = [tuple([int(pi) for pi in p]) for p in ps]
degs     = ps@grading
num_pfvs = [sum([np.all(pfv.pgrading[1:] == p) for pfv in pfvs]) for p in ps_safe]

In [ ]:
df = pd.DataFrame({
    'p': ps_safe,
    'deg': degs,
    'num_pfvs': num_pfvs
})

In [ ]:
degs_unique = sorted(set(df['deg']))
num_per_deg = []
avg_num_per_deg = []
for deg in degs_unique:
    df_tmp = df[df['deg']==deg]
    num_per_deg.append(sum(df_tmp['num_pfvs']))
    avg_num_per_deg.append(num_per_deg[-1]/len(df_tmp))

In [ ]:
N = 50

X = np.convolve(degs_unique, np.ones(N)/N, mode='valid')
Y = np.convolve(num_per_deg, np.ones(N)/N, mode='valid')
plt.scatter(X, Y)
plt.xlabel('p-vector degree')
plt.ylabel('# PFVs')
plt.title('Manwe; used moving average of size 50...')

In [ ]:
N = 50

X = np.convolve(degs_unique, np.ones(N)/N, mode='valid')
Y = np.convolve(avg_num_per_deg, np.ones(N)/N, mode='valid')
plt.scatter(X, Y)

plt.xlabel('p-vector degree')
plt.ylabel('avg # PFVs per p-vector')
plt.title('Manwe; used moving average of size 50...')

# Harvest

In [ ]:
W0s    = []
aligns = []
gsMs   = []

gvs = cy.compute_gv(max_deg=10).coo
for pfv in pfvs:
    pfv.gvs = gvs

    W0s.append(pfv.W0())
    aligns.append(pfv.align)
    gsMs.append(pfv.gsM)

In [ ]:
plt.scatter(W0s, aligns, c=gsMs)
plt.colorbar(label='gsM')
plt.yscale('log')
plt.xlabel('W0')
plt.ylabel('align')